1. Khởi tạo database trên Colab

In [ ]:
import sqlite3
import pandas as pd

# tạo database trong RAM
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

2. Tạo bảng

In [ ]:
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")

3. Insert dữ liệu

In [ ]:
students = [
(1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
(2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
(3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
(4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
(5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
(6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
(7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2)
]

courses = [
(12, 'Giai tich'),
(34, 'Thong ke'),
(26, 'Tin hoc')
]

cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", students)
cursor.executemany("INSERT INTO course VALUES (?, ?)", courses)

conn.commit()

4. JOIN các kiểu


In [ ]:
#INNER JOIN
pd.read_sql("""
SELECT *
FROM student s
INNER JOIN course c
ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


In [ ]:
#LEFT JOIN
pd.read_sql("""
SELECT *
FROM student s
LEFT JOIN course c
ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke


In [ ]:
#RIGHT JOIN
pd.read_sql("""
SELECT *
FROM course c
LEFT JOIN student s
ON s.course_id = c.id
""", conn)

,id,course_name,student_id,name,class,course_id,score
0,12,Giai tich,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,34,Thong ke,2.0,Tran Thi Lan,Kinh Te,34.0,9.2
2,34,Thong ke,7.0,Bui Tien Dung,Kinh Te,34.0,9.2
3,26,Tin hoc,NaN,None,None,NaN,NaN


In [ ]:
#FULL OUTER JOIN
pd.read_sql("""
SELECT *
FROM student s
LEFT JOIN course c ON s.course_id = c.id

UNION

SELECT *
FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn)

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,None,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34,9.2,34.0,Thong ke
7,12,Giai tich,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
8,26,Tin hoc,None,None,None,NaN,None
9,34,Thong ke,2,Tran Thi Lan,Kinh Te,34.0,9.2


5. Update course_id còn thiếu

In [ ]:
cursor.execute("""
UPDATE student
SET course_id = CASE
    WHEN class = 'Toan Tin' THEN 12
    WHEN class = 'Kinh Te' THEN 34
    WHEN class = 'May Tinh' THEN 26
END
WHERE course_id IS NULL
""")
conn.commit()

6. Xóa dữ liệu sai

In [ ]:
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()

7.Thống kê

In [ ]:
#a. theo lớp
pd.read_sql("""
SELECT class,
       COUNT(*) AS total_students,
       AVG(score) AS avg_score
FROM student
GROUP BY class
""", conn)

,class,total_students,avg_score
0,Kinh Te,2,9.2
1,May Tinh,1,6.7
2,Toan Tin,1,7.9


In [ ]:
#b. Theo môn học
pd.read_sql("""
SELECT c.course_name,
       COUNT(*) AS total_students,
       AVG(s.score) AS avg_score
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

,course_name,total_students,avg_score
0,Giai tich,2,7.3
1,Thong ke,2,9.2


In [ ]:
#c. Phân loại
pd.read_sql("""
SELECT c.course_name,
       AVG(s.score) AS avg_score,
       CASE
           WHEN AVG(s.score) >= 9 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6 THEN 'Tot'
           ELSE 'Kem'
       END AS classification
FROM student s
JOIN course c ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

,course_name,avg_score,classification
0,Giai tich,7.3,Tot
1,Thong ke,9.2,Xuat sac


8.Xếp hạng

In [ ]:
#a. Theo điểm
pd.read_sql("""
SELECT *,
       RANK() OVER (ORDER BY score DESC) AS rank_score
FROM student
""", conn)

,student_id,name,class,course_id,score,rank_score
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,4


In [ ]:
#b.Theo lớp
pd.read_sql("""
SELECT *,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_class
FROM student
""", conn)

,student_id,name,class,course_id,score,rank_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
3,3,Pham Van Nam,Toan Tin,12,7.9,1


In [ ]:
#c.Theo môn
pd.read_sql("""
SELECT *,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_course
FROM student
""", conn)

,student_id,name,class,course_id,score,rank_course
0,3,Pham Van Nam,Toan Tin,12,7.9,1
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
2,2,Tran Thi Lan,Kinh Te,34,9.2,1
3,7,Bui Tien Dung,Kinh Te,34,9.2,1


In [ ]:
#d. Top 3 cao nhất
pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score DESC) r
    FROM student
)
WHERE r <= 3
""", conn)

,student_id,name,class,course_id,score,r
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3


In [ ]:
e. Top 3 thấp nhất
pd.read_sql("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score ASC) r
    FROM student
)
WHERE r <= 3
""", conn)

,student_id,name,class,course_id,score,r
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,3,Pham Van Nam,Toan Tin,12,7.9,2
2,2,Tran Thi Lan,Kinh Te,34,9.2,3
3,7,Bui Tien Dung,Kinh Te,34,9.2,3


9. Thêm graduation_date

In [ ]:
cursor.execute("""
ALTER TABLE student
ADD COLUMN graduation_date TEXT
""")

In [ ]:
# Cập nhật ngày tốt nghiệp
cursor.execute("""
UPDATE student
SET graduation_date = datetime('now', '+' || (11 - score)*30 || ' days')
""")
conn.commit()

In [ ]:
# Xem kết quả
pd.read_sql("SELECT * FROM student", conn)

,student_id,name,class,course_id,score,graduation_date
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,None
1,2,Tran Thi Lan,Kinh Te,34,9.2,None
2,3,Pham Van Nam,Toan Tin,12,7.9,None
3,7,Bui Tien Dung,Kinh Te,34,9.2,None


In [ ]:
conn.close()